# Gold Layer — Star Schema (Python / Pandas version)
**CNG Distribution Analytics Platform**

Reads Silver **CSV** files and builds a Power BI–ready star schema:
three dimensions and three fact tables, written as **CSV** to the Gold
folder under `Files/`.

## Model
```
                 dim_date
                    |
   dim_customer — fact_sales — dim_product
                                   |
                 dim_product — fact_purchases
                                   |
                 dim_product — fact_inventory
```

| Table | Grain | Joins to |
|-------|-------|----------|
| `dim_product`   | one row per ProductID  | — |
| `dim_customer`  | one row per CustomerID | — |
| `dim_date`      | one row per calendar day | — |
| `fact_sales`    | one row per sales order line | product, customer, date |
| `fact_purchases`| one row per PO line | product, date |
| `fact_inventory`| one row per inventory snapshot line | product, date |

**Design choices**
- Joins use the **natural business keys** (ProductID, CustomerID, DateKey).
  They were cleaned/trimmed and de-duplicated in Silver, so no surrogate
  keys are needed for a star this size.
- Unmatched fact keys are routed to an **Unknown member** in each dimension
  (`PRD_UNKNOWN` / `CUST_UNKNOWN`) so rows are never silently dropped from
  Power BI visuals.
- `fact_sales.Division` is filled from the customer where missing.


## 1. Config

In [13]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────────
# FABRIC: notebook attached to a Lakehouse — read/write under Files/
SILVER_PATH = "abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/silverdata/"
GOLD_PATH   = "abfss://CNG_project@onelake.dfs.fabric.microsoft.com/gold_lakehouse.Lakehouse/Files/golddata/"

# LOCAL: uncomment these two lines instead
# SILVER_PATH = "data/output/silver/"
# GOLD_PATH   = "data/output/gold/"

os.makedirs(GOLD_PATH, exist_ok=True)

# Sentinel keys for Unknown dimension members
UNKNOWN_PRODUCT  = "PRD_UNKNOWN"
UNKNOWN_CUSTOMER = "CUST_UNKNOWN"

print(f"Silver path : {SILVER_PATH}")
print(f"Gold path   : {GOLD_PATH}")

Silver path : abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/silverdata/
Gold path   : abfss://CNG_project@onelake.dfs.fabric.microsoft.com/gold_lakehouse.Lakehouse/Files/golddata/


## 2. Imports

In [14]:
import pandas as pd
import numpy as np
from datetime import datetime

built_at = datetime.now()
print(f"pandas      : {pd.__version__}")
print(f"Run started : {built_at.strftime('%Y-%m-%d %H:%M:%S')}")

pandas      : 2.3.3
Run started : 2026-06-17 23:33:35


## 3. Load Silver
Dates were already cast in Silver, but CSV is text on disk — re-parse the
date columns we rely on here.

In [15]:
def load_silver(name: str, date_cols=None) -> pd.DataFrame:
    df = pd.read_csv(SILVER_PATH + f"silver_{name}.csv")
    for c in (date_cols or []):
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    # Silver audit column isn't needed downstream
    df = df.drop(columns=[c for c in ["_silver_transformed_at"] if c in df.columns])
    print(f"  {name:16s}: {len(df):,} rows")
    return df

print("Loading Silver tables:")
products  = load_silver("products",        ["IntroducedDate"])
customers = load_silver("customers",       ["CustomerSince"])
inventory = load_silver("inventory",       ["LastReceivedDate", "LastIssuedDate",
                                             "ExpiryDate", "LastUpdated"])
purchases = load_silver("purchase_orders", ["OrderDate", "ExpectedDeliveryDate",
                                             "ActualDeliveryDate"])
sales     = load_silver("sales_orders",    ["OrderDate", "RequestedDeliveryDate"])

Loading Silver tables:
  products        : 40 rows
  customers       : 70 rows
  inventory       : 187 rows
  purchase_orders : 1,144 rows
  sales_orders    : 4,864 rows


## 4. Dimensions

### 4.1  dim_product

In [16]:
def build_dim_product(products: pd.DataFrame) -> pd.DataFrame:
    cols = ["ProductID", "ProductName", "Category", "SubCategory", "UOM",
            "BasePrice_USD", "StandardCost_USD", "gross_margin_pct",
            "is_below_cost", "PrimarySupplierID", "Division",
            "WeightPerUnit_KG", "IsHazardous", "ActiveFlag", "IntroducedDate"]
    dim = products[[c for c in cols if c in products.columns]].copy()

    # Enforce one row per key (Silver already deduped, this is a guard)
    dim = dim.drop_duplicates(subset="ProductID", keep="first")

    # Append Unknown member for unmatched fact rows
    unknown = {c: np.nan for c in dim.columns}
    unknown["ProductID"]   = UNKNOWN_PRODUCT
    unknown["ProductName"] = "Unknown Product"
    unknown["Category"]    = "Unknown"
    dim = pd.concat([dim, pd.DataFrame([unknown])], ignore_index=True)

    print(f"  dim_product : {len(dim):,} rows "
          f"(incl. 1 Unknown member)")
    return dim

dim_product = build_dim_product(products)
dim_product.head(3)

/tmp/ipykernel_230/2543444337.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dim = pd.concat([dim, pd.DataFrame([unknown])], ignore_index=True)


,ProductID,ProductName,Category,SubCategory,UOM,BasePrice_USD,StandardCost_USD,gross_margin_pct,is_below_cost,PrimarySupplierID,Division,WeightPerUnit_KG,IsHazardous,ActiveFlag,IntroducedDate
0,PRD0001,Bleached Softwood Kraft Pulp,Pulp,Bskp,MT,680.0,802.64,-18.04,1.0,SUP0024,Lindenmeyr Publications,78.4,0.0,1.0,2018-04-07
1,PRD0002,Bleached Hardwood Kraft Pulp,Pulp,Bhkp,MT,620.0,474.02,23.55,0.0,SUP0024,Central National,NaN,0.0,1.0,2013-01-18
2,PRD0003,Fluff Pulp,Pulp,Fluff,MT,750.0,541.48,27.80,0.0,SUP0013,Central National,796.5,1.0,0.0,2013-12-11


### 4.2  dim_customer

In [17]:
def build_dim_customer(customers: pd.DataFrame) -> pd.DataFrame:
    cols = ["CustomerID", "CustomerName", "Segment", "Division", "Region",
            "Country", "State", "City", "AnnualRevenueTier", "CreditLimit_USD",
            "PaymentTerms", "PreferredCurrency", "TaxExempt",
            "SalesRepID", "AccountManagerID", "ActiveFlag", "CustomerSince"]
    dim = customers[[c for c in cols if c in customers.columns]].copy()

    dim = dim.drop_duplicates(subset="CustomerID", keep="first")

    unknown = {c: np.nan for c in dim.columns}
    unknown["CustomerID"]   = UNKNOWN_CUSTOMER
    unknown["CustomerName"] = "Unknown Customer"
    unknown["Segment"]      = "Unknown"
    unknown["Region"]       = "Unknown"
    dim = pd.concat([dim, pd.DataFrame([unknown])], ignore_index=True)

    print(f"  dim_customer: {len(dim):,} rows (incl. 1 Unknown member)")
    return dim

dim_customer = build_dim_customer(customers)
dim_customer.head(3)

/tmp/ipykernel_230/1851569358.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dim = pd.concat([dim, pd.DataFrame([unknown])], ignore_index=True)


,CustomerID,CustomerName,Segment,Division,Region,Country,State,City,AnnualRevenueTier,CreditLimit_USD,PaymentTerms,PreferredCurrency,TaxExempt,SalesRepID,AccountManagerID,ActiveFlag,CustomerSince
0,CUST00001,QUAD/GRAPHICS INC,Packaging Converter,Central National,AMER,USA,FL,New York,Platinum,1000000.0,Net 30,EUR,0.0,REP010,AM015,True,2000-06-01
1,CUST00002,RR Donnelley,Commercial Printer,North American Distribution,AMER,Mexico,NaN,Minneapolis,Bronze,10000000.0,Net 90,CAD,0.0,REP014,AM011,False,2003-11-07
2,CUST00003,LSC Communications,Foodservice,North American Distribution,AMER,united states,WA,NaN,Platinum,5000000.0,Net 30,GBP,0.0,REP017,AM014,True,2019-03-01


### 4.3  dim_date
Generated calendar spanning every date referenced by the facts, so Power BI
time-intelligence (YoY, MTD, etc.) works. `DateKey` is an integer `yyyymmdd`
— the facts carry the same key.

In [18]:
def build_dim_date(date_series_list) -> pd.DataFrame:
    # Gather min/max across all supplied date columns
    all_dates = pd.concat([s.dropna() for s in date_series_list])
    start = all_dates.min().normalize()
    end   = all_dates.max().normalize()
    # Pad to whole calendar years for clean time intelligence
    start = pd.Timestamp(year=start.year, month=1,  day=1)
    end   = pd.Timestamp(year=end.year,   month=12, day=31)

    rng = pd.date_range(start, end, freq="D")
    d = pd.DataFrame({"Date": rng})
    d["DateKey"]      = d["Date"].dt.strftime("%Y%m%d").astype(int)
    d["Year"]         = d["Date"].dt.year
    d["Quarter"]      = d["Date"].dt.quarter
    d["QuarterName"]  = "Q" + d["Quarter"].astype(str)
    d["Month"]        = d["Date"].dt.month
    d["MonthName"]    = d["Date"].dt.strftime("%B")
    d["MonthShort"]   = d["Date"].dt.strftime("%b")
    d["YearMonth"]    = d["Date"].dt.strftime("%Y-%m")
    d["Day"]          = d["Date"].dt.day
    d["DayOfWeek"]    = d["Date"].dt.dayofweek + 1          # 1=Mon
    d["DayName"]      = d["Date"].dt.strftime("%A")
    d["IsWeekend"]    = d["DayOfWeek"] >= 6
    d["Date"]         = d["Date"].dt.date                   # date-only for CSV

    print(f"  dim_date    : {len(d):,} rows ({start.date()} → {end.date()})")
    return d

dim_date = build_dim_date([
    sales["OrderDate"], sales["RequestedDeliveryDate"],
    purchases["OrderDate"], purchases["ExpectedDeliveryDate"],
    purchases["ActualDeliveryDate"],
    inventory["LastUpdated"],
])
dim_date.head(3)

  dim_date    : 1,826 rows (2021-01-01 → 2025-12-31)


,Date,DateKey,Year,Quarter,QuarterName,Month,MonthName,MonthShort,YearMonth,Day,DayOfWeek,DayName,IsWeekend
0,2021-01-01,20210101,2021,1,Q1,1,January,Jan,2021-01,1,5,Friday,False
1,2021-01-02,20210102,2021,1,Q1,1,January,Jan,2021-01,2,6,Saturday,True
2,2021-01-03,20210103,2021,1,Q1,1,January,Jan,2021-01,3,7,Sunday,True


## 5. Fact tables

In [19]:
def conform_key(fact: pd.DataFrame, key: str, valid_ids, unknown_value) -> pd.DataFrame:
    """
    Route fact rows whose key isn't in the dimension to the Unknown member,
    so nothing drops out of a Power BI relationship.
    """
    orphans = ~fact[key].isin(valid_ids)
    n = int(orphans.sum())
    if n:
        fact.loc[orphans, key] = unknown_value
    return fact, n


def add_date_key(fact: pd.DataFrame, date_col: str, new_col: str) -> pd.DataFrame:
    """Add an integer yyyymmdd DateKey from a datetime column (NaT -> <NA>)."""
    dk = fact[date_col].dt.strftime("%Y%m%d")
    fact[new_col] = pd.to_numeric(dk, errors="coerce").astype("Int64")
    return fact

### 5.1  fact_sales
Fills missing Division from the customer, conforms product/customer keys,
and stamps an `OrderDateKey` for the date relationship.

In [20]:
def build_fact_sales(sales, customers):
    f = sales.copy()

    # — Fill missing Division from the customer's Division —
    cust_div = (customers.drop_duplicates("CustomerID")
                          .set_index("CustomerID")["Division"])
    before_null = f["Division"].isna().sum()
    f["Division"] = f["Division"].fillna(f["CustomerID"].map(cust_div))
    after_null = f["Division"].isna().sum()
    print(f"  [sales] Division filled from customer: "
          f"{before_null - after_null} rows ({after_null} still unknown)")
    f["Division"] = f["Division"].fillna("Unknown")

    # — Conform keys to dimensions —
    f, n_prod = conform_key(f, "ProductID",  set(products["ProductID"]),  UNKNOWN_PRODUCT)
    f, n_cust = conform_key(f, "CustomerID", set(customers["CustomerID"]), UNKNOWN_CUSTOMER)
    print(f"  [sales] routed to Unknown — product: {n_prod}, customer: {n_cust}")

    # — Date key —
    f = add_date_key(f, "OrderDate", "OrderDateKey")

    # — Net amount (returns are negative qty already) —
    if "calc_line_total_usd" in f.columns:
        f["NetAmount_USD"] = np.where(
            f.get("is_return", False).astype(bool),
            -f["calc_line_total_usd"].abs(),
            f["calc_line_total_usd"]
        ).round(2)

    keep = ["OrderID", "OrderDateKey", "OrderDate", "CustomerID", "ProductID",
            "Division", "SalesChannel", "SalesRepID", "Status",
            "OrderedQty", "UnitPrice_USD", "DiscountPct", "discounted_unit_price",
            "LineTotal_USD", "calc_line_total_usd", "NetAmount_USD",
            "is_return", "IsRush", "RequestedDeliveryDate"]
    f = f[[c for c in keep if c in f.columns]]
    print(f"  fact_sales  : {len(f):,} rows")
    return f

fact_sales = build_fact_sales(sales, customers)
fact_sales.head(3)

  [sales] Division filled from customer: 0 rows (0 still unknown)
  [sales] routed to Unknown — product: 0, customer: 184
  fact_sales  : 4,864 rows


,OrderID,OrderDateKey,OrderDate,CustomerID,ProductID,Division,SalesChannel,SalesRepID,Status,OrderedQty,UnitPrice_USD,DiscountPct,discounted_unit_price,LineTotal_USD,calc_line_total_usd,NetAmount_USD,is_return,IsRush,RequestedDeliveryDate
0,SO0000001,20231230,2023-12-30,CUST00032,PRD0026,North American Distribution,Agent,REP005,On Hold,1247.0,2761.02,13.8,2379.9992,3442991.94,2967859.00,2967859.00,False,False,2024-01-03
1,SO0000002,20220925,2022-09-25,CUST00042,PRD0039,North American Distribution,Broker,REP008,Delivered,-1384.0,2776.93,10.5,2485.3524,3843271.12,3439727.72,-3439727.72,True,False,2022-10-15
2,SO0000003,20240620,2024-06-20,CUST00048,PRD0011,Central National,Inside Sales,REP009,Cancelled,305.0,2398.28,0.6,2383.8903,731475.40,727086.54,727086.54,False,True,2024-07-07


### 5.2  fact_purchases

In [21]:
def build_fact_purchases(purchases):
    f = purchases.copy()

    f, n_prod = conform_key(f, "ProductID", set(products["ProductID"]), UNKNOWN_PRODUCT)
    print(f"  [po] routed to Unknown — product: {n_prod}")

    f = add_date_key(f, "OrderDate", "OrderDateKey")

    keep = ["POID", "OrderDateKey", "OrderDate", "SupplierID", "ProductID",
            "Status", "ApprovalStatus", "Currency", "IncoTerms", "PortOfLoading",
            "OrderedQty", "ReceivedQty", "receipt_rate",
            "UnitCost_USD", "TotalCost_USD", "FreightCost_USD", "ExchangeRate",
            "ExpectedDeliveryDate", "ActualDeliveryDate",
            "expected_lead_time_days", "actual_lead_time_days", "is_late_delivery"]
    f = f[[c for c in keep if c in f.columns]]
    print(f"  fact_purchases: {len(f):,} rows")
    return f

fact_purchases = build_fact_purchases(purchases)
fact_purchases.head(3)

  [po] routed to Unknown — product: 0
  fact_purchases: 1,144 rows


,POID,OrderDateKey,OrderDate,SupplierID,ProductID,Status,ApprovalStatus,Currency,IncoTerms,PortOfLoading,...,receipt_rate,UnitCost_USD,TotalCost_USD,FreightCost_USD,ExchangeRate,ExpectedDeliveryDate,ActualDeliveryDate,expected_lead_time_days,actual_lead_time_days,is_late_delivery
0,PO000001,20231204,2023-12-04,SUP0029,PRD0028,Fully Received,Approved,CAD,FOB,Santos,...,1.0,1028.65,1734303.90,12546.69,0.8148,2023-12-22,2024-01-19,18.0,46.0,1.0
1,PO000002,20230816,2023-08-16,SUP0034,PRD0039,Fully Received,Approved,NOK,FOB,Yokohama,...,1.0,749.22,3718378.86,7009.87,1.1205,2023-11-05,2023-11-01,81.0,77.0,0.0
2,PO000003,20230603,2023-06-03,SUP0034,PRD0019,Cancelled,Approved,USD,NaN,Hamburg,...,0.0,329.23,1560550.20,12711.20,0.8378,2023-06-11,NaT,8.0,NaN,NaN


### 5.3  fact_inventory
Snapshot grain. `SnapshotDateKey` comes from `LastUpdated` so the snapshot
can sit on the date axis.

In [22]:
def build_fact_inventory(inventory):
    f = inventory.copy()

    f, n_prod = conform_key(f, "ProductID", set(products["ProductID"]), UNKNOWN_PRODUCT)
    print(f"  [inv] routed to Unknown — product: {n_prod}")

    f = add_date_key(f, "LastUpdated", "SnapshotDateKey")

    # Value at risk = value of stock sitting below reorder level
    if "BelowReorder" in f.columns and "TotalValue_USD" in f.columns:
        below = f["BelowReorder"].astype("boolean").fillna(False).astype(bool)
        f["ValueBelowReorder_USD"] = np.where(below, f["TotalValue_USD"], 0.0).round(2)

    keep = ["InventoryID", "SnapshotDateKey", "LastUpdated", "ProductID",
            "WarehouseID", "StorageLocation", "LotNumber",
            "StockQty", "AllocatedQty", "AvailableQty",
            "ReorderLevel", "ReorderQty", "BelowReorder",
            "UnitCost_USD", "TotalValue_USD", "ValueBelowReorder_USD",
            "LastReceivedDate", "LastIssuedDate", "ExpiryDate"]
    f = f[[c for c in keep if c in f.columns]]
    print(f"  fact_inventory: {len(f):,} rows")
    return f

fact_inventory = build_fact_inventory(inventory)
fact_inventory.head(3)

  [inv] routed to Unknown — product: 0
  fact_inventory: 187 rows


,InventoryID,SnapshotDateKey,LastUpdated,ProductID,WarehouseID,StorageLocation,LotNumber,StockQty,AllocatedQty,AvailableQty,ReorderLevel,ReorderQty,BelowReorder,UnitCost_USD,TotalValue_USD,ValueBelowReorder_USD,LastReceivedDate,LastIssuedDate,ExpiryDate
0,INV000001,20241021,2024-10-21,PRD0001,WH015,Rack-F27,LOT888595,7442.0,0.0,7442.0,244.0,1220.0,False,465.43,3463730.06,0.0,2024-02-28,2024-07-05,NaT
1,INV000002,20241105,2024-11-05,PRD0001,WH018,Rack-G40,LOT937297,950.0,3.0,947.0,250.0,1000.0,False,1620.56,1539532.00,0.0,2024-11-22,2024-12-07,NaT
2,INV000003,20241107,2024-11-07,PRD0001,WH016,Rack-B20,LOT550512,1766.0,194.0,1572.0,246.0,738.0,False,2473.44,4368095.04,0.0,2024-09-24,2024-06-11,NaT


## 6. Write Gold
Writes CSV to the Gold `Files/` folder.
Swap this one function when migrating back to Spark/Delta in Fabric.

In [23]:
def write_gold(df: pd.DataFrame, name: str):
    """
    PYTHON VERSION — writes CSV to Gold output folder under Files/.

    SPARK VERSION  — replace body with:
        spark_df = spark.createDataFrame(df)
        spark_df.write.format('delta').mode('overwrite') \\
            .option('overwriteSchema', 'true').saveAsTable(name)
    """
    df = df.copy()
    df["_gold_built_at"] = str(built_at)
    out_path = GOLD_PATH + name + ".csv"
    df.to_csv(out_path, index=False)
    print(f"  {name:16s}: {len(df):,} rows → {out_path}")


gold_tables = {
    "dim_product"    : dim_product,
    "dim_customer"   : dim_customer,
    "dim_date"       : dim_date,
    "fact_sales"     : fact_sales,
    "fact_purchases" : fact_purchases,
    "fact_inventory" : fact_inventory,
}

print("Writing Gold tables:")
for name, df in gold_tables.items():
    write_gold(df, name)

Writing Gold tables:
  dim_product     : 41 rows → abfss://CNG_project@onelake.dfs.fabric.microsoft.com/gold_lakehouse.Lakehouse/Files/golddata/dim_product.csv
  dim_customer    : 71 rows → abfss://CNG_project@onelake.dfs.fabric.microsoft.com/gold_lakehouse.Lakehouse/Files/golddata/dim_customer.csv
  dim_date        : 1,826 rows → abfss://CNG_project@onelake.dfs.fabric.microsoft.com/gold_lakehouse.Lakehouse/Files/golddata/dim_date.csv
  fact_sales      : 4,864 rows → abfss://CNG_project@onelake.dfs.fabric.microsoft.com/gold_lakehouse.Lakehouse/Files/golddata/fact_sales.csv
  fact_purchases  : 1,144 rows → abfss://CNG_project@onelake.dfs.fabric.microsoft.com/gold_lakehouse.Lakehouse/Files/golddata/fact_purchases.csv
  fact_inventory  : 187 rows → abfss://CNG_project@onelake.dfs.fabric.microsoft.com/gold_lakehouse.Lakehouse/Files/golddata/fact_inventory.csv


## 7. Star-schema validation
Confirms the model is sound **before** you wire it up in Power BI:
unique dimension keys, and every fact key present in its dimension.

In [24]:
problems = []

# 1. Dimension keys must be unique (else relationships break / double-count)
for dim, key in [(dim_product, "ProductID"),
                 (dim_customer, "CustomerID"),
                 (dim_date, "DateKey")]:
    dupes = dim[key].duplicated().sum()
    status = "OK " if dupes == 0 else "FAIL"
    if dupes: problems.append(f"{key} has {dupes} duplicate(s)")
    print(f"  [{status}] {key} unique in dimension ({dupes} dupes)")

# 2. Every fact key must resolve to a dimension member
checks = [
    ("fact_sales.ProductID",      fact_sales["ProductID"],    set(dim_product["ProductID"])),
    ("fact_sales.CustomerID",     fact_sales["CustomerID"],   set(dim_customer["CustomerID"])),
    ("fact_sales.OrderDateKey",   fact_sales["OrderDateKey"], set(dim_date["DateKey"])),
    ("fact_purchases.ProductID",  fact_purchases["ProductID"], set(dim_product["ProductID"])),
    ("fact_purchases.OrderDateKey", fact_purchases["OrderDateKey"], set(dim_date["DateKey"])),
    ("fact_inventory.ProductID",  fact_inventory["ProductID"], set(dim_product["ProductID"])),
    ("fact_inventory.SnapshotDateKey", fact_inventory["SnapshotDateKey"], set(dim_date["DateKey"])),
]
for label, series, valid in checks:
    # ignore NA keys (e.g. rows with no date) — Power BI handles blanks
    missing = (~series.dropna().isin(valid)).sum()
    status = "OK " if missing == 0 else "FAIL"
    if missing: problems.append(f"{label}: {missing} unmatched")
    print(f"  [{status}] {label}: {missing} unmatched")

print("\n" + ("✅ Star schema valid — ready for Power BI."
      if not problems else f"❌ {len(problems)} issue(s): " + "; ".join(problems)))

  [OK ] ProductID unique in dimension (0 dupes)
  [OK ] CustomerID unique in dimension (0 dupes)
  [OK ] DateKey unique in dimension (0 dupes)
  [OK ] fact_sales.ProductID: 0 unmatched
  [OK ] fact_sales.CustomerID: 0 unmatched
  [OK ] fact_sales.OrderDateKey: 0 unmatched
  [OK ] fact_purchases.ProductID: 0 unmatched
  [OK ] fact_purchases.OrderDateKey: 0 unmatched
  [OK ] fact_inventory.ProductID: 0 unmatched
  [OK ] fact_inventory.SnapshotDateKey: 0 unmatched

✅ Star schema valid — ready for Power BI.


## 8. Power BI modelling notes
After loading the six CSVs into Power BI:

**Relationships (all single-direction, one-to-many from dim → fact):**
- `dim_product[ProductID]`  → `fact_sales[ProductID]`, `fact_purchases[ProductID]`, `fact_inventory[ProductID]`
- `dim_customer[CustomerID]` → `fact_sales[CustomerID]`
- `dim_date[DateKey]` → `fact_sales[OrderDateKey]`, `fact_purchases[OrderDateKey]`, `fact_inventory[SnapshotDateKey]`

Because there are three date relationships into three different facts,
each works as an active relationship on its own fact. If you later need
two date roles on the *same* fact (e.g. order vs. delivery), add a second
inactive relationship and activate it with `USERELATIONSHIP` in the measure.

- Mark `dim_date` as the official **Date table** (`Date` column).
- Hide the raw `*DateKey` columns on the facts from report view.